## Expert Lineage Extracter Worker

### An automated pipeline that leverages a Large Language Model (LLM) tuned as a Senior Data Architect to:

Ingest: Scan RSQL files from a central repository.

Analyze: Identify source-to-target flows, transformation logic (Upserts/Full Reloads), and key impact columns.

Standardize: Output Collibra-compatible JSON structures.

Visualize: Generate instant lineage diagrams for technical review.

This first implementation will use a simplistic, brute-force type of RAG..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications -AI-Driven Automated Metadata Harvesting: Synchronizing Redshift ETL Lineage with Collibra Governance.</h2>
            <span style="color:#181;">In the current data landscape, maintaining accurate data lineage is critical for regulatory compliance, impact analysis, and data trust. Manually documenting lineage for complex RSQL (Redshift) ETL processes is error-prone and time-consuming. This project proposes an Automated Lineage Extraction Framework using Generative AI (LangChain) to programmatically parse SQL scripts and synchronize metadata with Collibra..</span>
        </td>
    </tr>
</table>

In [ ]:
import os
import glob
import json
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# Define the Schema for Collibra-Compatible Lineage
class LineageStep(BaseModel):
    operation: str = Field(description="The SQL operation: COPY, DELETE, INSERT, etc.")
    source_name: str = Field(description="The source table or S3 path")
    target_name: str = Field(description="The destination table")
    logic: str = Field(description="Short description of the transformation logic")

class LineageFull(BaseModel):
    script_name: str
    lineage: List[LineageStep]

In [5]:
SYSTEM_PROMPT_MESSAGE1 = """

You are a Senior Data Architect specializing in Metadata Management and Collibra Data Governance. 
Your task is to parse RSQL (Amazon Redshift) ETL scripts and generate a technical data lineage map 
in a structured JSON format.

### OBJECTIVE
Analyze the provided SQL code to identify every movement of data. You must distinguish between external 
sources (S3), staging areas (Temp Tables), and final production targets.

### EXTRACTION RULES
1. SOURCE IDENTIFICATION: Look for 'COPY' commands to identify S3 buckets, manifests, and file formats (e.g., PARQUET).
2. STAGING IDENTIFICATION: Identify 'CREATE TEMP TABLE' or 'CREATE TABLE' statements that serve as intermediate hops.
3. TARGET IDENTIFICATION: Identify the final table receiving data via 'INSERT INTO' or 'MERGE'.
4. TRANSFORMATION LOGIC: 
   - If you see a 'DELETE' followed by an 'INSERT' on the same table using a shared key, label the operation as "UPSERT".
   - If you see a 'TRUNCATE' followed by an 'INSERT', label it as "FULL_RELOAD".
5. ATTRIBUTE MAPPING: Capture join keys (e.g., sales_id) and filter conditions.

### OUTPUT JSON SCHEMA
Your output must be a valid JSON object following this structure:
{
  "script_metadata":{ "name": "string", "dialect": "Redshift/RSQL" },
  "lineage": [
    {
      "step_id": integer,
      "operation": "COPY|INSERT|DELETE|UPSERT|MAINTENANCE",
      "source": { "name": "string", "type": "S3_BUCKET|TABLE|VIEW" },
      "target": { "name": "string", "type": "TABLE|TEMP_TABLE" },
      "logic": "Detailed description of the SQL transformation",
      "impact_columns": ["list", "of", "keys"]
    }
  ]
}

### CONSTRAINTS
- Do not include administrative SQL like 'SET' or 'ECHO' in the lineage steps.
- Ensure the 'target' of one step correctly matches the 'source' of the subsequent step to maintain a continuous chain.
- Provide only the JSON output; do not include conversational filler.

"""

In [5]:
SYSTEM_PROMPT_MESSAGE = """
You are a Senior Data Architect specializing in Metadata Management and Collibra Data Governance. 
Your task is to parse RSQL (Amazon Redshift) ETL scripts and generate a technical data lineage map 
in a structured JSON format.

### COMPLEXITY HANDLING
1. NON-SQL CONTENT: The input may contain RSQL-specific meta-commands (e.g., \\set, \\echo, \\if, \\gset). You must ignore these for the lineage map UNLESS they define variables used in the SQL.
2. DYNAMIC PARAMETERS: The SQL contains variables (prefixed with ':'). You will be provided with a parameter context. You MUST replace these placeholders with their actual values (e.g., replace ':v_s3_path' with 's3://my-bucket/data/') to ensure the lineage reflects physical locations.

### OBJECTIVE
Analyze the provided SQL code to identify every movement of data. You must distinguish between external 
sources (S3), staging areas (Temp Tables), and final production targets.

### EXTRACTION RULES
1. SOURCE IDENTIFICATION: Look for 'COPY' commands to identify S3 buckets, manifests, and file formats (e.g., PARQUET).
2. STAGING IDENTIFICATION: Identify 'CREATE TEMP TABLE' or 'CREATE TABLE' statements that serve as intermediate hops.
3. TARGET IDENTIFICATION: Identify the final table receiving data via 'INSERT INTO' or 'MERGE'.
4. ATTRIBUTE MAPPING: Capture join keys (e.g., sales_id) and filter conditions.

### OUTPUT JSON SCHEMA
Your output must be a valid JSON object following this structure:
{{
  "script_metadata": {{ "name": "string", "dialect": "Redshift/RSQL" }},
  "lineage": [
    {{
      "step_id": "integer",
      "operation": "COPY|INSERT|DELETE|UPSERT|MAINTENANCE",
      "source": {{ "name": "string", "type": "S3_BUCKET|TABLE|VIEW" }},
      "target": {{ "name": "string", "type": "TABLE|TEMP_TABLE" }},
      "logic": "Detailed description of the SQL transformation",
      "impact_columns": ["list", "of", "keys"]
    }}
  ]
}}

### CONSTRAINTS
- Do not include administrative SQL like 'SET' or 'ECHO' in the lineage steps.
- Ensure the 'target' of one step correctly matches the 'source' of the subsequent step to maintain a continuous chain.
- Provide only the JSON output; do not include conversational filler.
"""

In [7]:
input_folder="sql_container/etl/*.sql"
rSqlFilenames = glob.glob(input_folder)
rSqlFiles={}
if not rSqlFilenames:
    print("No SQL files found.")
else:
    print(rSqlFilenames)
    for rSqlFilename in rSqlFilenames:
        name = Path(rSqlFilename).stem
        with open(rSqlFilename, "r", encoding="utf-8") as f:
            rSqlFiles[name.lower()] = f.read()


['sql_container/etl\\sample_etl_work.sql']


In [8]:
USER_PROMPT_MESSAGE = """
Analyze this SQL script and extract the lineage steps: \n\n {sql_code}
"""

In [9]:
output_json="sql_container/output/"
llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm = llm.with_structured_output(LineageFull)
prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT_MESSAGE),
        ("user", USER_PROMPT_MESSAGE)
    ])
chain = prompt | structured_llm

for rSqlFile in rSqlFiles:
    lineage_data = chain.invoke({"sql_code": rSqlFiles[rSqlFile]})
    out_file=output_json+rSqlFile+".json"
    with open(out_file, 'w') as f:
        json.dump(lineage_data.dict(), f, indent=2)
    print(f"Lineage saved to {output_json}")
    
   

Lineage saved to sql_container/output/


C:\Users\Samrat\AppData\Local\Temp\ipykernel_7580\1254290724.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  json.dump(lineage_data.dict(), f, indent=2)


In [10]:
from graphviz import Digraph

def draw_lineage(lineage_data: LineageFull, output_image: str):
    # Initialize graph with a clean horizontal flow
    dot = Digraph(comment='Data Lineage', graph_attr={'rankdir': 'LR', 'nodesep': '0.5', 'ranksep': '1.0'})
    dot.attr('node', fontname='Helvetica', fontsize='10', shape='none')

    # Standardized color palette for data operations
    op_colors = {
        "COPY": "#2ecc71",       # Green for ingestion
        "UPSERT": "#3498db",     # Blue for transformation
        "INSERT": "#9b59b6",     # Purple for loading
        "DELETE": "#e74c3c",     # Red for removal
        "MAINTENANCE": "#e67e22" # Orange for optimization
    }

    # Tracking unique nodes to avoid re-declaring them
    nodes = set()

    for step in lineage_data.lineage:
        # Create professional HTML-styled nodes for Source and Target
        for name in [step.source_name, step.target_name]:
            if name not in nodes:
                # Detect type based on name for visual hinting
                bg_color = "#f0f0f0" if "s3://" in name.lower() else "#e1f5fe"
                label = f'''<
                    <TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" CELLPADDING="8">
                        <TR><TD BGCOLOR="{bg_color}"><B>{name}</B></TD></TR>
                    </TABLE>>'''
                dot.node(name, label)
                nodes.add(name)

        # Draw the connection using the specific operation name
        color = op_colors.get(step.operation.upper(), "#7f8c8d")
        dot.edge(
            step.source_name, 
            step.target_name, 
            label=f" {step.operation} ",
            color=color,
            fontcolor=color,
            penwidth="2.0"
        )

    dot.render(output_image, format='png', cleanup=True)
    print(f"Diagram successfully generated: {output_image}.png")

In [11]:
draw_lineage(lineage_data, "output_image")

Diagram successfully generated: output_image.png



## COMPLEXITY HANDLING
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/voyage.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
        
- NON-SQL CONTENT: The input may contain RSQL-specific meta-commands (e.g., \\set, \\echo, \\if, \\gset). You must ignore these for the lineage map UNLESS they define variables used in the SQL.

- DYNAMIC PARAMETERS: The SQL contains variables (prefixed with ':'). You will be provided with a parameter context. You MUST replace these placeholders with their actual values (e.g., replace ':v_s3_path' with 's3://my-bucket/data/') to ensure the lineage reflects physical locations.
</td>
</tr>
</table>

In [ ]:
param_context=""
sql_content=""

USER_PROMPT_MESSAGE = f"""
    PARAMETER CONTEXT (Variable Values):
    {param_context}

    RSQL SCRIPT TO ANALYZE:
    {sql_content}
    """

In [32]:
input_folder="sql_container/etl/*.sql"
rSqlFilenames = glob.glob(input_folder)
rSqlFiles={}
conFiles={}
if not rSqlFilenames:
    print("No SQL files found.")
else:
    print(rSqlFilenames)

for rSqlFilename in rSqlFilenames:
    print("rSqlFilename >>>>>>>>>",rSqlFilename)
    confFileName=rSqlFilename.replace(".sql",".conf")
    print("confFileName>>>>>>>>>>>",confFileName)
    name = Path(rSqlFilename).stem
    print("name after stem",name)
    with open(rSqlFilename, "r", encoding="utf-8") as f:
        rSqlFiles[name.lower()] = f.read()
    # 2. Read the Parameter file (e.g., a .txt or .env file)
    with open(confFileName, 'r', encoding="utf-8") as f:
        print("###############",name.lower())
        conFiles[name.lower()] = f.read()
        print(conFiles[name.lower()])





['sql_container/etl\\engine_upsert.sql']
rSqlFilename >>>>>>>>> sql_container/etl\engine_upsert.sql
confFileName>>>>>>>>>>> sql_container/etl\engine_upsert.conf
name after stem engine_upsert
############### engine_upsert
# Global Environment Settings
DB_NAME="production_dw"
CLUSTER_ID="dw-cluster-01"
REGION="us-east-1"

# Specific Job Configuration (Metadata Driven)
export SRC_BUCKET="s3://corp-datalake-prod/raw/erp_system/"
export STG_TABLE="stg_orders_incremental"
export TGT_TABLE="fct_orders"
export PK_COLUMNS="order_id, line_item_id"
export DIST_KEY="order_id"
export SORT_KEY="order_date"

# ETL Control Variables
export LOAD_TIMESTAMP=$(date +'%Y-%m-%d %H:%M:%S')
export LAST_SUCCESSFUL_LOAD=$(cat .last_load_timestamp) # High-watermark


In [37]:
output_json="sql_container/output/"

llm = ChatOpenAI(model="gpt-5.2", temperature=0)
structured_llm = llm.with_structured_output(LineageFull)
prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT_MESSAGE),
        ("user", "{USER_PROMPT_MESSAGE}")
    ])
chain = prompt | structured_llm

for rSqlFile in rSqlFiles:
    user_input = f"""
    Analyze this Parameter Context, RSQL script and extract the lineage steps: 

    #PARAMETER CONTEXT (Variable Values):
    #{conFiles[rSqlFile]}

    #RSQL SCRIPT TO ANALYZE:
    #{rSqlFiles[rSqlFile]}
    
    """
    lineage_data = chain.invoke({"USER_PROMPT_MESSAGE":user_input})
    out_file=output_json+rSqlFile+".json"
    with open(out_file, 'w') as f:
        json.dump(lineage_data.dict(), f, indent=2)
    print(f"Lineage saved to {output_json}")
    
   

Lineage saved to sql_container/output/


C:\Users\Samrat\AppData\Local\Temp\ipykernel_11532\1996051473.py:25: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  json.dump(lineage_data.dict(), f, indent=2)


In [43]:
import tempfile
try:
    import networkx as nx
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

def draw_lineage_graph(lineage_data: LineageFull) -> str | None:
    """Render lineage as a graph image. Returns path to temporary PNG or None if not available."""
    if not HAS_PLOTTING or not lineage_data.lineage:
        return None

    G = nx.DiGraph()
    for step in lineage_data.lineage:
        G.add_edge(
            step.source_name,
            step.target_name,
            operation=step.operation,
        )

    if G.number_of_edges() == 0:
        return None

    fig, ax = plt.subplots(figsize=(12, 6))
    pos = nx.spring_layout(G, k=2.5, seed=42)
    node_colors = []
    for n in G.nodes():
        if "s3://" in n.lower() or "s3:" in n.lower():
            node_colors.append("#2ecc71")  # green for S3
        elif "temp" in n.lower() or "stage" in n.lower() or "stg" in n.lower():
            node_colors.append("#3498db")  # blue for staging
        else:
            node_colors.append("#9b59b6")  # purple for target

    edge_colors = {
        "COPY": "#2ecc71",
        "INSERT": "#9b59b6",
        "DELETE": "#e74c3c",
        "UPSERT": "#3498db",
        "MAINTENANCE": "#e67e22",
    }
    edge_color_list = [
        edge_colors.get(e[2]["operation"].upper(), "#7f8c8d")
        for e in G.edges(data=True)
    ]

    nx.draw_networkx_nodes(
        G, pos, node_color=node_colors, node_size=2000,
        alpha=0.9, ax=ax, node_shape="s"
    )
    nx.draw_networkx_labels(
        G, pos, font_size=8, ax=ax,
        labels={n: n if len(n) <= 35 else n[:32] + "..." for n in G.nodes()}
    )
    nx.draw_networkx_edges(
        G, pos, edge_color=edge_color_list, width=2,
        arrows=True, arrowsize=20, ax=ax,
        connectionstyle="arc3,rad=0.1"
    )
    edge_labels = {(u, v): d["operation"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8, ax=ax)

    ax.set_title("Data Lineage", fontsize=14, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()

    out_path = "sample_diagram.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight", facecolor="white")
    plt.close()
    return out_path

In [44]:
out_file=draw_lineage_graph(lineage_data)

In [41]:
out_file

'C:\\Users\\Samrat\\AppData\\Local\\Temp\\tmpcbehjr1s.png'

In [ ]:
# Initialize Gemini 1.5 Pro
# Ensure your GOOGLE_API_KEY is set in environment variables
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)
structured_llm = llm.with_structured_output(LineageFull)